# L19 demo: a hand-rolled, tool-using agent

This notebook builds a tool-using agent loop from scratch, no framework, against
three real tools: a read-only query over the L3/L4 Intel Lab sensor store, a
stats helper, and a call into a surrogate model in the spirit of L13's airfoil
noise surrogate. The loop, the tools, and the harness (budgets, error recovery,
loop detection, logging) are all real and run end to end below.

**One honest substitution.** There is no hosted-LLM call in this notebook: this
environment has no API key and no network path to a model provider. The "model"
driving the loop below is a small, deterministic, scripted stand-in that returns
tool calls in exactly the shape a real tool-calling API would, so every mechanic
this session cares about (the message loop, tool dispatch, budgets, error
recovery, loop-breaking, logging) is genuinely exercised. What a scripted stand-in
cannot show you is a model actually *deciding*; A10 is where you connect this same
harness to a real hosted model and find out whether it decides well.

**One honest simplification.** `call_surrogate` below is not L13's fitted Gaussian
process; it is a small analytic function with the same five inputs and the same
validated-range checking, built so this notebook does not depend on downloading
the UCI airfoil dataset again. It is deliberately *not* built around a strong
velocity-power law, because L13's notes measured that exact law directly against
the real data and found it does not hold for this dataset's scaled SPL column;
a stand-in that assumed it anyway would contradict a finding this course already
verified.


## The sensor store: the same Intel Lab data, queried read-only

Same source as L3 and L4: 2004 Intel Berkeley Lab motes, temperature, humidity,
light, and voltage. We rebuild the Parquet file here so this notebook does not
depend on another lecture's notebook having already run.

In [ ]:
import io
import zipfile
import urllib.request
from pathlib import Path

import pandas as pd

CACHE = Path('.cache')
CACHE.mkdir(exist_ok=True)
PARQUET = CACHE / 'readings.parquet'
URL = 'https://raw.githubusercontent.com/linsea423/Intel_Lab_Data/master/data.zip'
COLS = ['date', 'time', 'epoch', 'moteid', 'temperature', 'humidity', 'light', 'voltage']

if not PARQUET.exists():
    print('downloading', URL)
    with urllib.request.urlopen(URL) as r:
        payload = r.read()
    with zipfile.ZipFile(io.BytesIO(payload)) as z:
        with z.open('data.txt') as f:
            raw = pd.read_csv(f, sep=r'\s+', header=None, names=COLS, on_bad_lines='skip')

    df = raw.dropna(subset=['moteid']).copy()
    df['moteid'] = pd.to_numeric(df['moteid'], errors='coerce')
    df = df.dropna(subset=['moteid'])
    df['moteid'] = df['moteid'].astype(int)
    df = df[df.moteid.between(1, 54)]
    df['ts'] = pd.to_datetime(df['date'] + ' ' + df['time'], format='mixed', errors='coerce')
    df = df.dropna(subset=['ts'])
    for c in ['temperature', 'humidity', 'light', 'voltage']:
        df[c] = pd.to_numeric(df[c], errors='coerce')
    df = df.dropna(subset=['temperature', 'humidity', 'light', 'voltage'])
    readings = df[['moteid', 'ts', 'temperature', 'humidity', 'light', 'voltage']].sort_values(['moteid', 'ts'])
    readings.to_parquet(PARQUET, index=False)

print('ready:', PARQUET)


## Tool 1: `query_sensor_db`, read-only

The single most important property of a data tool an agent can call: it cannot
write. This function only ever runs a `SELECT`. Notice the error path returns a
plain, informative dictionary rather than raising, so a caller (model or harness)
gets something to react to instead of a crashed loop. Some mote ids genuinely have
no data in this real dataset (a mote can die mid-deployment), which is a more
honest error case to design around than a made-up one.

In [ ]:
import duckdb

con = duckdb.connect(':memory:')
ALLOWED_VARS = {'temperature', 'humidity', 'light', 'voltage'}

def query_sensor_db(mote_id: int, variable: str, limit: int = 200) -> dict:
    """Read-only: the most recent `limit` readings of `variable` for one mote."""
    if variable not in ALLOWED_VARS:
        return {'error': f'unknown variable {variable!r}; allowed: {sorted(ALLOWED_VARS)}'}
    rows = con.execute(
        f"SELECT {variable} AS value FROM read_parquet(?) "
        "WHERE moteid = ? ORDER BY ts DESC LIMIT ?",
        [str(PARQUET), mote_id, limit],
    ).fetchall()
    if not rows:
        return {'error': f'no rows found for mote_id={mote_id}. Not every id 1-54 has data.'}
    return {'mote_id': mote_id, 'variable': variable, 'n': len(rows), 'values': [r[0] for r in rows]}

query_sensor_db(1, 'voltage', limit=5)


## Tool 2: `compute_stats`

The simplest tool in the set, and worth including anyway: an agent that has to
call an LLM again just to average a list of numbers is paying latency and cost
for arithmetic a function already does exactly and instantly.

In [ ]:
import numpy as np

def compute_stats(values: list) -> dict:
    if not values:
        return {'error': 'empty series'}
    arr = np.asarray(values, dtype=float)
    return {'mean': float(arr.mean()), 'std': float(arr.std()),
            'min': float(arr.min()), 'max': float(arr.max()), 'n': len(arr)}

compute_stats([1.0, 2.0, 3.0])


## Tool 3: `call_surrogate`

Five inputs, matching L13's airfoil self-noise surrogate exactly, and the same
validated input ranges from that dataset's documentation (frequency 200-20,000 Hz,
angle of attack 0-22.2 degrees, chord 0.0254-0.3048 m, free-stream velocity
31.7-71.3 m/s, displacement thickness 0.0004-0.058 m). An input outside that range
is not extrapolation the surrogate should silently attempt, it is a request the
tool should refuse and say so, the same "check before you trust it" discipline
L13 spent a whole section on.

In [ ]:
import math

SURROGATE_RANGES = {
    'freq_hz': (200.0, 20000.0),
    'aoa_deg': (0.0, 22.2),
    'chord_m': (0.0254, 0.3048),
    'velocity_ms': (31.7, 71.3),
    'thickness_m': (0.0004, 0.058),
}

def call_surrogate(freq_hz: float, aoa_deg: float, chord_m: float,
                    velocity_ms: float, thickness_m: float) -> dict:
    args = dict(freq_hz=freq_hz, aoa_deg=aoa_deg, chord_m=chord_m,
                velocity_ms=velocity_ms, thickness_m=thickness_m)
    out_of_range = [f'{k}={v} outside {SURROGATE_RANGES[k]}'
                     for k, v in args.items() if not (SURROGATE_RANGES[k][0] <= v <= SURROGATE_RANGES[k][1])]
    if out_of_range:
        return {'error': "input outside the surrogate's validated range: " + '; '.join(out_of_range)}
    # A small analytic stand-in, not L13's fitted GP -- see the notebook intro for why.
    spl = (110.0
           + 5.0 * math.log10(velocity_ms / 50.0)
           + 0.15 * (aoa_deg - 8.0) ** 2
           - 8.0 * math.log10(chord_m / 0.1)
           + 2.0 * math.log10(freq_hz / 2000.0))
    return {'spl_db': round(spl, 2)}

print(call_surrogate(2000, 8, 0.1, 50, 0.01))
print(call_surrogate(2000, 8, 0.1, 5, 0.01))   # out of range: on purpose


## Unit-test the tools, independently of any model

This is the "determinism and testing" topic from the notes made concrete: every
tool is a plain Python function, so it gets tested the way any function does,
with no model in the loop at all. If a tool is wrong, you want to know that from
`pytest`, not from watching an agent behave strangely.

In [ ]:
def test_tools():
    assert query_sensor_db(999999, 'voltage') == {
        'error': 'no rows found for mote_id=999999. Not every id 1-54 has data.'}
    assert query_sensor_db(1, 'pressure')['error'].startswith('unknown variable')
    assert compute_stats([])['error'] == 'empty series'
    assert compute_stats([2.0, 4.0]) == {'mean': 3.0, 'std': 1.0, 'min': 2.0, 'max': 4.0, 'n': 2}
    assert 'error' in call_surrogate(2000, 8, 0.1, 5, 0.01)   # velocity too low
    assert 'spl_db' in call_surrogate(2000, 8, 0.1, 50, 0.01)

test_tools()
print('all tool unit tests passed')


## Tool schemas: what the model actually reads

Below is the exact shape a real tool-calling API expects: a name, a description
written for the *model* to decide when to call it, and a JSON Schema for typed,
validated arguments. Vague wording here is a specification bug, not a model
limitation; "get sensor data" invites the wrong arguments in a way "the most
recent N readings of one variable for one mote, read-only" does not.

In [ ]:
TOOL_SCHEMAS = [
    {
        'name': 'query_sensor_db',
        'description': ('Read-only lookup of the most recent N readings of one variable '
                         '(temperature, humidity, light, or voltage) for one mote id (1-54, '
                         'some ids have no data). Returns an error if the mote or variable is invalid.'),
        'input_schema': {
            'type': 'object',
            'properties': {
                'mote_id': {'type': 'integer', 'minimum': 1, 'maximum': 54},
                'variable': {'type': 'string', 'enum': sorted(ALLOWED_VARS)},
                'limit': {'type': 'integer', 'minimum': 1, 'maximum': 1000, 'default': 200},
            },
            'required': ['mote_id', 'variable'],
        },
    },
    {
        'name': 'compute_stats',
        'description': 'Mean, standard deviation, min, and max of a list of numbers.',
        'input_schema': {
            'type': 'object',
            'properties': {'values': {'type': 'array', 'items': {'type': 'number'}}},
            'required': ['values'],
        },
    },
    {
        'name': 'call_surrogate',
        'description': ('Predict airfoil sound pressure level (dB) from five operating '
                         'parameters. Returns an error if any input is outside the '
                         "surrogate's validated training range."),
        'input_schema': {
            'type': 'object',
            'properties': {k: {'type': 'number', 'minimum': lo, 'maximum': hi}
                            for k, (lo, hi) in SURROGATE_RANGES.items()},
            'required': list(SURROGATE_RANGES),
        },
    },
]
[t['name'] for t in TOOL_SCHEMAS]


## The agent loop, concretely

Send messages and tool definitions to the model. The model returns either a tool
call or a final answer. If it is a tool call, the harness executes it (never the
model) and appends the result as a new message. Repeat until the model produces a
final answer, the model repeats an identical failing call (a stuck loop), or the
step budget runs out. Every step is logged.

In [ ]:
TOOLS_IMPL = {'query_sensor_db': query_sensor_db, 'compute_stats': compute_stats, 'call_surrogate': call_surrogate}

def run_agent(task, model_step, max_steps=6):
    """model_step(messages, step_number) -> {'tool_call': {...}} or {'final_answer': str}"""
    messages = [{'role': 'user', 'content': task}]
    trace = [{'step': 0, 'role': 'user', 'content': task}]
    last_signature, repeats = None, 0

    for step in range(1, max_steps + 1):
        action = model_step(messages, step)

        if action.get('final_answer') is not None:
            trace.append({'step': step, 'role': 'assistant', 'content': action['final_answer']})
            return {'status': 'done', 'answer': action['final_answer'], 'trace': trace, 'steps': step}

        call = action['tool_call']
        trace.append({'step': step, 'role': 'assistant', 'content': {'tool_call': call}})

        signature = (call['name'], tuple(sorted(call['arguments'].items())))
        repeats = repeats + 1 if signature == last_signature else 0
        last_signature = signature
        if repeats >= 2:
            trace.append({'step': step, 'role': 'harness',
                          'content': f'loop-break: {call["name"]} called with identical arguments 3 times in a row'})
            return {'status': 'loop_break', 'trace': trace, 'steps': step}

        try:
            result = TOOLS_IMPL[call['name']](**call['arguments'])
        except Exception as e:
            result = {'error': f'{type(e).__name__}: {e}'}   # never let a tool crash the loop
        trace.append({'step': step, 'role': 'tool_result', 'content': result})
        messages.append({'role': 'assistant', 'content': {'tool_call': call}})
        messages.append({'role': 'tool', 'content': result})

    return {'status': 'budget_exhausted', 'trace': trace, 'steps': max_steps}

def print_trace(result):
    print('status:', result['status'], '| steps:', result['steps'])
    for t in result['trace']:
        print(f"  [{t['step']}] {t['role']}: {str(t['content'])[:110]}")


## A scripted stand-in for the model

`happy_path_model` plays a fixed plan, in the exact request/response shape a real
tool-calling model would use: look up mote 1's recent voltage, summarize it, map
it (crudely, and said so) onto the surrogate's velocity range, then sweep angle of
attack and report the setting with the lowest predicted noise. This is the
ReAct-style pattern from the notes: observe a result, decide the next action,
repeat, stop when the goal is met.

In [ ]:
def voltage_to_velocity(mean_voltage):
    """Toy mapping only: rescale a nominal 2.0-3.0 V battery range onto the
    surrogate's 31.7-71.3 m/s velocity range. Not a physical relationship."""
    lo, hi = SURROGATE_RANGES['velocity_ms']
    frac = max(0.0, min(1.0, (mean_voltage - 2.0) / (3.0 - 2.0)))
    return round(lo + frac * (hi - lo), 1)

def happy_path_model():
    mem = {}
    def step(messages, n):
        if n == 1:
            return {'tool_call': {'name': 'query_sensor_db',
                                   'arguments': {'mote_id': 1, 'variable': 'voltage', 'limit': 20}}}
        if n == 2:
            values = messages[-1]['content']['values']
            return {'tool_call': {'name': 'compute_stats', 'arguments': {'values': values}}}
        if n == 3:
            stats = messages[-1]['content']
            mem['stats'] = stats
            mem['velocity'] = voltage_to_velocity(stats['mean'])
            mem['aoa_grid'] = [2, 8, 14, 20]
            mem['results'] = []
            aoa = mem['aoa_grid'].pop(0)
            mem['cur_aoa'] = aoa
            return {'tool_call': {'name': 'call_surrogate', 'arguments': {
                'freq_hz': 2000.0, 'aoa_deg': float(aoa), 'chord_m': 0.1,
                'velocity_ms': mem['velocity'], 'thickness_m': 0.01}}}
        prev = messages[-1]['content']
        mem['results'].append((mem['cur_aoa'], prev.get('spl_db')))
        if mem['aoa_grid']:
            aoa = mem['aoa_grid'].pop(0)
            mem['cur_aoa'] = aoa
            return {'tool_call': {'name': 'call_surrogate', 'arguments': {
                'freq_hz': 2000.0, 'aoa_deg': float(aoa), 'chord_m': 0.1,
                'velocity_ms': mem['velocity'], 'thickness_m': 0.01}}}
        best_aoa, best_spl = min(mem['results'], key=lambda t: t[1])
        return {'final_answer': (
            f"Mote 1's recent voltage averaged {mem['stats']['mean']:.3f} V, mapped to a toy "
            f"velocity of {mem['velocity']} m/s. Sweeping angle of attack over "
            f"{[a for a, _ in mem['results']]} degrees, the lowest predicted noise is "
            f"{best_spl} dB at aoa={best_aoa} deg.")}
    return step

print_trace(run_agent("Find the operating point that minimizes predicted noise, "
                       "using mote 1's recent readings to set the velocity.",
                       happy_path_model(), max_steps=8))


## Bounding the loop: a hard step budget

The same task, same model, but told it has 2 steps instead of 8. It stops
cleanly with `budget_exhausted` rather than running forever or crashing. A real
agent should return whatever partial progress it made, not silence.

In [ ]:
print_trace(run_agent("Find the operating point that minimizes predicted noise, "
                       "using mote 1's recent readings to set the velocity.",
                       happy_path_model(), max_steps=2))


## Error recovery: a mote that does not exist

Mote 5 genuinely has no data in this dataset (a real dropout, not a made-up
error). Watch the model observe the tool's error message and recover by trying a
different mote instead of the loop dying on the first failure.

In [ ]:
def error_recovery_model():
    def step(messages, n):
        if n == 1:
            return {'tool_call': {'name': 'query_sensor_db',
                                   'arguments': {'mote_id': 5, 'variable': 'voltage', 'limit': 20}}}
        if n == 2:
            assert 'error' in messages[-1]['content']   # observed the failure
            return {'tool_call': {'name': 'query_sensor_db',
                                   'arguments': {'mote_id': 1, 'variable': 'voltage', 'limit': 20}}}
        values = messages[-1]['content']['values']
        return {'final_answer': f"mote 5 had no data; used mote 1 instead, mean voltage "
                                 f"{sum(values) / len(values):.3f} V"}
    return step

print_trace(run_agent("Get mote 5's voltage.", error_recovery_model(), max_steps=6))


## Bounding the loop: detecting a stuck model

This scripted model never adapts: it asks for mote 5 every time, regardless of
the error it keeps getting back, which is exactly the "model repeats a failing
call" failure the notes warn about. The harness notices the third identical call
and breaks the loop itself rather than burning the rest of the budget on a call
that is never going to succeed.

In [ ]:
def stubborn_model():
    def step(messages, n):
        return {'tool_call': {'name': 'query_sensor_db',
                               'arguments': {'mote_id': 5, 'variable': 'voltage', 'limit': 20}}}
    return step

print_trace(run_agent("Get mote 5's voltage.", stubborn_model(), max_steps=6))


Four runs, four different `status` values (`done`, `budget_exhausted`,
`loop_break`), and every one of them a harness decision, not a hope. None of this
required a real model to demonstrate, because none of it is about whether the
model is smart; it is about whether the code around the model behaves when the
model is not. A10 asks you to point this exact harness at a real hosted model and
find out.

Full notes, with the planning-pattern and guardrail material this notebook only
exercises rather than explains: [`../notes.md`](notes.md).